# DL_T05_1B: Paddy leaf disease detection on a low-cost device
Team 5, Task 1, Variant 1B (Paddy Doctor dataset). CM701PC / CM751PC Deep Learning.

**How to use**
1. Copy this whole project folder to Google Drive as `MyDrive/DL_T05_1B/` (the `src/` folder must be inside it).
2. Runtime > Change runtime type > T4 GPU. Run cells top to bottom.
3. `RUN_TRAINING = False` (default) loads the saved models and only re-computes evaluations (about 15-30 min).
   Set it to `True` to retrain everything (about 2-3 hours). Retraining gives similar but **not identical** numbers.

**Status:** the code in this notebook was assembled from the cells used for the reported results and refactored into `src/`.
The modules were smoke-tested on random data. The notebook as a whole must be run once by the team before you say it reproduces the numbers in the README.

In [ ]:
%pip install -q kaggle opencv-python-headless seaborn
import os, sys, json, time, random, pathlib, zipfile, subprocess, getpass
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from google.colab import drive
drive.mount("/content/drive")

PROJECT = pathlib.Path("/content/drive/MyDrive/DL_T05_1B")
for sub in ["outputs", "models", "splits", "cache"]:
    (PROJECT / sub).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT / "src"))
assert (PROJECT / "src" / "models.py").exists(), "Copy the src/ folder into MyDrive/DL_T05_1B/"

from config import SEED, IMG
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

RUN_TRAINING = False   # True = retrain E1-E7 (slow)
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1. Data: download (only if needed), stratified split with leakage check, image cache

In [ ]:
from data import build_table, make_splits, build_arrays
SPLITS, CACHE = PROJECT / "splits", PROJECT / "cache"
names = ["train", "val", "test"]
have_splits = all((SPLITS / f"{n}.csv").exists() for n in names) and (SPLITS / "classes.json").exists()
have_cache = all((CACHE / f"{k}_{n}.npy").exists() for k in "Xy" for n in names)

all_df = None
if not (have_splits and have_cache):
    # Join the competition on Kaggle first (accept the rules), then paste your credentials.
    os.environ["KAGGLE_USERNAME"] = input("Kaggle username: ").strip()
    os.environ["KAGGLE_KEY"] = getpass.getpass("Kaggle API key: ").strip()
    DATA = pathlib.Path("/content/data"); DATA.mkdir(exist_ok=True)
    subprocess.run(["kaggle", "competitions", "download", "-c", "paddy-disease-classification",
                    "-p", str(DATA)], check=True)
    with zipfile.ZipFile(DATA / "paddy-disease-classification.zip") as z:
        z.extractall(DATA / "paddy")
    all_df = build_table(DATA / "paddy" / "train_images")

if have_splits:                                   # the saved split is the authoritative one
    train, val, test = [pd.read_csv(SPLITS / f"{n}.csv") for n in names]
    classes = json.load(open(SPLITS / "classes.json"))
else:
    train, val, test, classes, info = make_splits(all_df, SEED)
    print("Leakage check:", info)
    for n, d in zip(names, (train, val, test)):
        d.to_csv(SPLITS / f"{n}.csv", index=False)
    json.dump(classes, open(SPLITS / "classes.json", "w"))
    all_df.to_csv(SPLITS / "all_images.csv", index=False)

class_to_idx = {c: i for i, c in enumerate(classes)}
NUM_CLASSES = len(classes)
rc_path = SPLITS / "rare_classes.json"
if rc_path.exists():
    rare_classes = json.load(open(rc_path))["rare_classes"]
else:   # the two smallest classes are the "rare" classes
    full = all_df if all_df is not None else pd.concat([train, val, test])
    rare_classes = list(full["label"].value_counts().sort_values().index[:2])
    json.dump({"rare_classes": rare_classes, "seed": SEED}, open(rc_path, "w"))
print("Rare classes:", rare_classes)

if not have_cache:
    for n, d in zip(names, (train, val, test)):
        X, y = build_arrays(d, IMG)
        np.save(CACHE / f"X_{n}.npy", X); np.save(CACHE / f"y_{n}.npy", y)
X_train, y_train = np.load(CACHE / "X_train.npy"), np.load(CACHE / "y_train.npy")
X_val, y_val = np.load(CACHE / "X_val.npy"), np.load(CACHE / "y_val.npy")
X_test, y_test = np.load(CACHE / "X_test.npy"), np.load(CACHE / "y_test.npy")
print(X_train.shape, X_val.shape, X_test.shape)   # reported run: (7252,...) (1559,...) (1562,...)

## 2. Exploratory analysis: class balance and the two rare classes

In [ ]:
counts = pd.DataFrame({"train": train["label"].value_counts(), "val": val["label"].value_counts(),
                       "test": test["label"].value_counts()}).loc[classes]
counts["total"] = counts.sum(axis=1)
print(counts.sort_values("total"))
order = counts["total"].sort_values()
plt.figure(figsize=(9, 4))
plt.barh(order.index, order.values, color=["crimson" if c in rare_classes else "steelblue" for c in order.index])
plt.xlabel("images (after duplicate removal)"); plt.title("Paddy Doctor class sizes (rare classes in red)")
plt.tight_layout(); plt.savefig(PROJECT / "outputs" / "class_distribution.png", dpi=150); plt.show()

## 3. Models, augmentation and the experiment runner (all experiments are judged on VALIDATION only)

In [ ]:
from models import build_scratch_cnn, build_mobilenet
from augment import make_tf_augment, make_ds, FieldAug, build_e7
from train import Experiment
from evaluate import metrics_on, full_report, tflite_convert, tflite_predict, latency_ms
from shift import make_shift_set

exp = Experiment(X_train, y_train, X_val, y_val, classes, rare_classes,
                 PROJECT / "outputs" / "experiment_log.csv", augment=make_tf_augment(SEED), seed=SEED)
print({classes[i]: round(w, 2) for i, w in exp.class_weights.items()})

## 4. Experiments E1 to E7 (only when `RUN_TRAINING = True`)
| ID | What | Question |
|---|---|---|
| E1 | scratch CNN, seeds 1-3 | baseline and run-to-run variance |
| E2/E3 | MobileNetV2-0.35 frozen head, then fine-tuned (E3 recipe: heavier geometric augmentation, 15 epochs) | transfer learning, frozen vs fine-tuned |
| E4 | as E3 without augmentation | does augmentation help? |
| E5 | as E4 without class weights | do class weights protect rare classes? |
| E6 | as E4 with width 0.5 | is a bigger model worth it? |
| E7 | milder geometry + blur/noise/low-res augmentation inside the model, 30 epochs, patience 6 | robustness to field-like degradation |

E7 differs from E4 in augmentation, epoch cap and patience; the effect of each was not isolated.

In [ ]:
def run_recipe(name, alpha=0.35, augment_on=True, use_cw=True, save_as=None):
    m = build_mobilenet(IMG, NUM_CLASSES, alpha)
    exp.fit(name + " (head)", m, 10, 1e-3, augment_on, use_cw)
    m.get_layer("backbone").trainable = True
    exp.fit(name, m, 15, 1e-4, augment_on, use_cw)
    if save_as: m.save(PROJECT / "models" / save_as)
    return m

if RUN_TRAINING:
    for s in [1, 2, 3]:                                   # E1: explicit seeds
        tf.keras.utils.set_random_seed(s)
        exp.fit(f"E1 scratch CNN seed {s}", build_scratch_cnn(IMG, NUM_CLASSES), epochs=25, lr=1e-3)
    tf.keras.utils.set_random_seed(SEED)
    run_recipe("E3 MobileNetV2-0.35 + fine-tune (aug, cw)", save_as="mnet035_E3.keras")
    run_recipe("E4 no-augmentation", augment_on=False, save_as="E4.keras")
    run_recipe("E5 no-aug no-class-weights", augment_on=False, use_cw=False)
    run_recipe("E6 no-aug width-0.5", alpha=0.5, augment_on=False)
    # E7: augmentation (incl. blur/noise/low-res) lives INSIDE the model, so tf.data augmentation is off
    w7, m7 = build_e7(build_mobilenet(IMG, NUM_CLASSES, 0.35), IMG, SEED)
    exp.fit("E7 field-aug (head)", w7, 10, 1e-3, augment_on=False)
    m7.get_layer("backbone").trainable = True
    exp.fit("E7 field-aug", w7, 30, 1e-4, augment_on=False, patience=6)
    m7.save(PROJECT / "models" / "E7.keras")              # inference model only
else:
    print("RUN_TRAINING=False: showing the saved experiment log")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
pd.DataFrame(exp.results).round(3)

## 5. Field-shift proxy sets (SIMULATED, not real field photographs)
Real field photos could not be collected within the available time and resources, so code-generated degradations are applied to held-out images.
Set A overlaps with E7's training degradations; set B uses different types and parameters (only low light is truly unrelated).
Seeds: validation A = SEED, B = SEED+1; test A = SEED+2, B = SEED+3 (test sets were never used for model selection).

In [ ]:
m3 = tf.keras.models.load_model(PROJECT / "models" / "mnet035_E3.keras")
m4 = tf.keras.models.load_model(PROJECT / "models" / "E4.keras")
m7 = tf.keras.models.load_model(PROJECT / "models" / "E7.keras")

X_val_A, kind_A = make_shift_set(X_val, "A", SEED)
X_val_B, kind_B = make_shift_set(X_val, "B", SEED + 1)

fig, axes = plt.subplots(2, 6, figsize=(16, 5.5))
for j in range(6):
    axes[0, j].imshow(X_val[j]); axes[0, j].axis("off"); axes[0, j].set_title("clean")
    axes[1, j].imshow(X_val_B[j]); axes[1, j].axis("off"); axes[1, j].set_title(kind_B[j])
plt.tight_layout(); plt.savefig(PROJECT / "outputs" / "shiftB_examples.png", dpi=150); plt.show()

rows = []
for nm, mdl in [("E3 (aug, 15 ep)", m3), ("E4 (no aug)", m4), ("E7 (field aug)", m7)]:
    for cond, X in [("clean val", X_val), ("shift A (in-family)", X_val_A), ("shift B (different types)", X_val_B)]:
        rows.append({"model": nm, "data": cond, **metrics_on(mdl, X, y_val, classes, rare_classes)})
cmp_df = pd.DataFrame(rows); cmp_df.to_csv(PROJECT / "outputs" / "shift_comparison_E3_E4_E7.csv", index=False)
print(cmp_df.to_string())

## 6. Evidence for Q2: training vs validation vs shifted accuracy (un-augmented training images)

In [ ]:
rows = []
for nm, mdl in [("E3", m3), ("E4", m4), ("E7", m7)]:
    rows.append({"model": nm,
                 "train_acc": metrics_on(mdl, X_train, y_train, classes, rare_classes)["acc"],
                 "val_acc": metrics_on(mdl, X_val, y_val, classes, rare_classes)["acc"],
                 "val_shiftA_acc": metrics_on(mdl, X_val_A, y_val, classes, rare_classes)["acc"],
                 "val_shiftB_acc": metrics_on(mdl, X_val_B, y_val, classes, rare_classes)["acc"]})
q2 = pd.DataFrame(rows); q2.to_csv(PROJECT / "outputs" / "q2_gaps.csv", index=False); print(q2.to_string())

## 7. Final test evaluation of the chosen model (E7)
Decision rule fixed BEFORE running E7: E7 replaces E4 only if val macro F1 >= 0.80, both rare recalls >= 0.75, and macro F1 on shift B is at least 0.05 above E4. It passed, so E7 is the final model.
The test split is used only for this final model. Do not switch models after looking at these numbers.

In [ ]:
X_test_A, _ = make_shift_set(X_test, "A", SEED + 2)
X_test_B, _ = make_shift_set(X_test, "B", SEED + 3)
summary = []
for tag, X in [("test_clean", X_test), ("test_shiftA_proxy", X_test_A), ("test_shiftB_proxy", X_test_B)]:
    rep = full_report(m7, X, y_test, "E7_" + tag, classes, str(PROJECT / "outputs"), title="E7 confusion matrix")
    summary.append({"data": tag, "accuracy": round(rep["accuracy"], 3),
                    "macroF1": round(rep["macro avg"]["f1-score"], 3),
                    **{f"recall_{c}": round(rep[c]["recall"], 3) for c in rare_classes}})
summary_df = pd.DataFrame(summary); summary_df.to_csv(PROJECT / "outputs" / "final_E7_summary.csv", index=False)
print(summary_df.to_string())

## 8. TensorFlow Lite conversion: size, single-thread latency, accuracy
Latency is model inference only (no JPEG decoding or resizing) on the Colab CPU, so it is a proxy for an Android phone, not a phone measurement.

In [ ]:
rows = []
for mode in ["float32", "dynamic"]:
    b = tflite_convert(m7, mode)
    (PROJECT / "models" / f"paddy_E7_{mode}.tflite").write_bytes(b)
    med, p95 = latency_ms(b, X_val)
    pred = tflite_predict(b, X_test)
    from sklearn.metrics import f1_score, recall_score
    rec = recall_score(y_test, pred, average=None, labels=np.arange(NUM_CLASSES))
    rows.append({"variant": mode, "size_MB": round(len(b) / 1e6, 2),
                 "latency_median_ms": round(med, 1), "latency_p95_ms": round(p95, 1),
                 "test_macroF1": round(f1_score(y_test, pred, average="macro"), 4),
                 **{f"recall_{c}": round(rec[classes.index(c)], 3) for c in rare_classes}})
tfl = pd.DataFrame(rows); tfl.to_csv(PROJECT / "outputs" / "tflite_E7.csv", index=False); print(tfl.to_string())

## 9. Scorecard against the PDF targets

In [ ]:
t = summary_df.set_index("data").loc["test_clean"]
sizes = tfl.set_index("variant")
print("Size < 10 MB         :", sizes["size_MB"].to_dict(), "->", bool((sizes["size_MB"] < 10).all()))
print("Latency < 300 ms     :", sizes["latency_median_ms"].to_dict(), "->", bool((sizes["latency_median_ms"] < 300).all()), "(Colab CPU proxy)")
print("Macro F1 >= 0.80     :", t["macroF1"], "->", t["macroF1"] >= 0.80)
for c in rare_classes:
    print(f"Recall {c} >= 0.75:", t[f"recall_{c}"], "->", t[f"recall_{c}"] >= 0.75)

## 10. Package the project (excludes the 1.5 GB image cache)

In [ ]:
zip_path = pathlib.Path("/content/DL_T05_1B.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in PROJECT.rglob("*"):
        if p.is_file() and "cache" not in p.relative_to(PROJECT).parts and "__pycache__" not in p.parts:
            z.write(p, pathlib.Path("DL_T05_1B") / p.relative_to(PROJECT))
print("ZIP size MB:", round(zip_path.stat().st_size / 1e6, 1))
from google.colab import files; files.download(str(zip_path))